# Clinicopathological and Molecular Characteristics of Second Primary Colorectal Cancer in Cancer Survivors including MSI-H Status and Anatomical Distribution Exploration with `mlcroissant`
This notebook provides a demonstration for loading and exploring the FAIR² dataset using the [`mlcroissant`](https://github.com/mlcommons/croissant) library.

### Dataset Source
The dataset is described by a Croissant schema available at:

`https://sen.science/doi/10.71728/senscience.qs2f-h81p/fair2.json`

In [ ]:
# Install mlcroissant if not already installed
!pip install mlcroissant

## 1. Data Loading
Load the dataset metadata using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd
import pprint

# Dataset Croissant schema URL
croissant_url = "https://sen.science/doi/10.71728/senscience.qs2f-h81p/fair2.json"

# Load the dataset package
dataset = mlc.Dataset(croissant_url)
# Display dataset metadata summary
print(f"Dataset name: {dataset.metadata.name}\n")
print("Description:")
print(dataset.metadata.description)
print("\nCite as:")
print(dataset.metadata.cite_as)


## 2. Data Overview
Review available record sets (`@id`), their fields (`@id`), and the first few records.

The FAIR² Croissant schema may contain multiple record sets. Here, we'll enumerate the available record sets by their `@id` and print details about their fields, referencing all elements by their `@id` as required.

In [ ]:
# List all record sets in the dataset, showing their @id and constituent fields/columns
record_sets = dataset.record_sets
print(f"Found {len(record_sets)} record sets in the dataset.")

for rs in record_sets:
    print("\nRecord set @id:", rs.id)
    print("  Name: ", rs.name)
    print("  Fields (by @id):")
    for fld in rs.fields:
        print(f"    - {fld.id}")
    if hasattr(rs, "columns") and rs.columns:
        print("  Columns (by @id):")
        for col in rs.columns:
            print(f"    - {col.id}")

> **Example records preview**

Let's print the first 2 records from each available record set, referencing the entities by their `@id`.

**Note:** Replace `<record_set_id>` in subsequent cells with one of the printed record set `@id`s.

In [ ]:
# Preview the first 2 records from each record set by @id
print('Sample records from each record set:')
for rs in record_sets:
    print(f"\nRecord set @id: {rs.id}")
    try:
        for i, rec in enumerate(dataset.records(record_set=rs.id)):
            pprint.pprint(rec)
            if i>=1:
                break
    except Exception as e:
        print(f"  Unable to preview records: {e}")

## 3. Data Extraction

Load data from a **specific record set** into a pandas DataFrame for analysis. Use the record set and field `@id`s from the overview above.

Below, we'll extract *all* record sets into their own DataFrames, with the keys in the Python dictionary being each record set's `@id`.

In [ ]:
# Extract data from each record set to pandas DataFrame
dataframes = {}
for rs in record_sets:
    records = list(dataset.records(record_set=rs.id))
    df = pd.DataFrame(records)
    dataframes[rs.id] = df
    print(f"Record set @id: {rs.id} - DataFrame shape: {df.shape}")
    print(f"  Columns (@id): {df.columns.tolist()}\n")

> **Choose a record set for further analysis**

Let's pick the main clinical data table for further analysis.

Below, set the variable `chosen_record_set_id` to the `@id` of the most relevant record set for EDA (use the one with the clinical records).

In [ ]:
# Set the @id of the clinical data record set (Replace with actual @id printed above if needed)
# If unsure, check output of previous cells for available record set @id. We'll use the first if only one is present.
if record_sets:
    chosen_record_set_id = record_sets[0].id
    print(f"Using record set: {chosen_record_set_id}")
else:
    raise ValueError("No record sets found in the schema.")

df = dataframes[chosen_record_set_id]
print(df.head(3))

## 4. Exploratory Data Analysis (EDA)
We'll walk through example analyses:
- Filtering records by a numeric field
- Normalizing numeric values
- Aggregating by a grouping field

All fields and columns must be referenced using their `@id` as shown in the DataFrame columns.

In [ ]:
# List all column @id in the selected record set
print("All columns (@id) in the selected DataFrame:")
for col in df.columns:
    print(f"- {col}")

Let's select a **numeric field or column** from the above. Replace `<numeric_field_id>` below with the appropriate `@id` (e.g., for 'age_at_diagnosis' or similar).

In [ ]:
# Replace with @id of a numeric column (consult previous cell output)
# Example: numeric_field_id = 'age_at_diagnosis'  # Use actual @id from above
numeric_field_id = None
for col in df.columns:
    # This example attempts to auto-select the first numeric field
    if df[col].dtype.kind in 'ifc' and df[col].notnull().sum() > 0:
        numeric_field_id = col
        break
# If not found, try to find a likely numeric column by name
if not numeric_field_id:
    for col in df.columns:
        if 'age' in str(col).lower() or 'interval' in str(col).lower() or 'years' in str(col).lower():
            numeric_field_id = col
            break
if not numeric_field_id:
    raise ValueError("Could not auto-select a numeric field; please set numeric_field_id manually.")
print(f"Using numeric field: {numeric_field_id} (@id)")

# Set threshold for filtering (choose appropriate based on field)
if df[numeric_field_id].dtype.kind in 'fi':
    threshold = df[numeric_field_id].quantile(0.4)
else:
    threshold = 10  # fallback default

filtered_df = df[df[numeric_field_id] > threshold].copy()
print(f"Filtered {len(filtered_df)} records with {numeric_field_id} > {threshold}")
print(filtered_df[[numeric_field_id]].head())

# Normalize field
filtered_df[f"{numeric_field_id}_normalized"] = (filtered_df[numeric_field_id] - filtered_df[numeric_field_id].mean()) / filtered_df[numeric_field_id].std()
print(f"\nNormalization example for {numeric_field_id}:")
print(filtered_df[[numeric_field_id, f"{numeric_field_id}_normalized"]].head())

# Try grouping by a likely categorical field (@id)
print('\nColumns available for grouping:')
for col in df.columns:
    print(f'- {col}')

# Auto-select a likely group field (replace as needed)
group_field_id = None
for col in df.columns:
    if col != numeric_field_id and df[col].dtype == 'object':
        unique = df[col].nunique()
        if 1 < unique <= 6:
            group_field_id = col
            break

if not group_field_id:
    print("No suitable group field found; skipping group aggregation.")
else:
    print(f"\nGrouping by field: {group_field_id} (@id)")
    grouped_df = filtered_df.groupby(group_field_id)[numeric_field_id].mean().to_frame()
    print("Grouped means:")
    print(grouped_df.head())

## 5. Visualization
Visualize data distributions and relationships between fields. All axes/labels should reference fields by their `@id`.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

# Histogram of the numeric field
plt.figure(figsize=(7,4))
sns.histplot(df[numeric_field_id].dropna(), bins=12, kde=True)
plt.title(f"Distribution of {numeric_field_id} (@id)")
plt.xlabel(numeric_field_id)
plt.ylabel("Count")
plt.show()

# If group_field_id is found, create a boxplot by group
if group_field_id:
    plt.figure(figsize=(8,5))
    sns.boxplot(data=df, x=group_field_id, y=numeric_field_id)
    plt.title(f"{numeric_field_id} by {group_field_id} (@id)")
    plt.ylabel(numeric_field_id)
    plt.xlabel(group_field_id)
    plt.show()

## 6. Conclusion

- The FAIR² dataset was successfully loaded and explored using `mlcroissant` using only entity references specified by their `@id`.
- We examined record set and field structure, selected specific fields for analysis, and visualized their distributions.
- Next steps could include statistical modeling or outcome prediction based on the clinical variables, ensuring reference to the Croissant schema throughout.

> **Tip:** To analyze other columns or record sets, repeat above steps referencing their `@id`.